# EMET2007 Week 11: Unit Root Tests and Forecast Evaluation

## Learning Objectives

By the end of this tutorial, you will be able to:
1. Understand the distinction between stationary and non-stationary time series
2. Conduct Augmented Dickey-Fuller (ADF) unit root tests
3. Interpret unit root test results correctly
4. Implement pseudo out-of-sample forecast evaluation
5. Calculate and interpret forecast errors

---

## Introduction

Continuing our analysis of Australian CPI and inflation, we now focus on **trending behaviour**:
- Does the series have a **deterministic trend** (predictable upward/downward movement)?
- Does it have a **stochastic trend** (unit root, random walk behaviour)?

This matters because non-stationary series require different statistical treatment and can lead to spurious regression results.

---

## Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import adfuller

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/juergenmeinecke/EMET2007/refs/heads/main/datasets/cpi_aus_2026.csv')

In [ ]:
df['date'] = pd.to_datetime(df['time'], format='%b-%Y')
df.index = pd.DatetimeIndex(df.date, name='quarter').to_period('Q')

### Exercise 1: Recreate Variables from Last Week

Run the cells below to recreate `logcpi` and `infl` from Week 10.

In [ ]:
df['logcpi'] = np.log(df.cpi)
df['infl'] = 400 * df.logcpi.diff()

---

## Part 1: Restricting the Sample Period

### Exercise 2: Australian Monetary Policy History

A brief history relevant to our analysis:
- **Until early 1970s:** Fixed exchange rate
- **Late 1983:** Australian Dollar floated
- **Early 1990s:** Inflation targeting adopted

**Decision:** Restrict our analysis to 1984:Q1 onwards to:
1. Have a consistent policy regime
2. Still have enough observations for reliable estimation

In [ ]:
# Plot full inflation series to see regime changes
# fig, ax = plt.subplots(figsize=(10, 4))
# ax.plot(df.date, df.infl)
# ax.axvline(pd.to_datetime('1984-01-01'), color='red', linestyle='--', label='1984Q1')
# ax.set_xlabel('Time')
# ax.set_ylabel('Inflation (%)')
# ax.set_title('Australian Inflation - Full History')
# ax.legend()
# plt.show()

In [ ]:
# Create restricted dataset from 1984 onwards
# df_1984 = df[df.index >= '1984Q1'].copy()
# print(f'Sample: {df_1984.index[0]} to {df_1984.index[-1]} ({len(df_1984)} obs)')

---

## Part 2: Unit Root Tests

### Background: The Dickey-Fuller Test

To test if a series $Y_t$ has a unit root, we estimate:

$$\Delta Y_t = \beta_0 + \alpha \cdot t + \delta Y_{t-1} + u_t$$

And test:
- $H_0: \delta = 0$ (unit root exists, series is non-stationary)
- $H_1: \delta < 0$ (no unit root, series is stationary)

**Important:** The test statistic does NOT follow a standard t-distribution under the null. Use special critical values.

### Exercise 3: Test CPI for Unit Root

**Two methods:**

1. **Manual:** Run the regression and examine the t-statistic on $Y_{t-1}$
2. **Using `adfuller`:** Built-in function with correct critical values

In [ ]:
# Create variables for the manual Dickey-Fuller regression
# df_1984['diff_cpi'] = df_1984.cpi.diff()
# df_1984['l1cpi'] = df_1984.cpi.shift(1)
# df_1984['trend'] = range(1, len(df_1984) + 1)

In [ ]:
# Manual unit root regression for CPI:
# reg_cpi = smf.ols('diff_cpi ~ trend + l1cpi', data=df_1984, missing='drop').fit()
# reg_cpi.summary()
# Check the t-statistic on l1cpi — compare to ADF critical values, not standard t-table

In [ ]:
# Automated ADF test for CPI (equivalent to the manual regression above)
# adf_cpi = adfuller(df_1984.cpi, maxlag=0, regression='ct')
# print(f'Test statistic: {adf_cpi[0]:.4f}')
# print(f'p-value:        {adf_cpi[1]:.4f}')
# print(f'Critical values: {adf_cpi[4]}')

**Interpretation:** If p-value > 0.05, we cannot reject the null of a unit root.

*What does this mean for CPI?*

### Exercise 4: Test Inflation for Unit Root

In [ ]:
# Create lag of inflation for manual regression
# df_1984['diff_infl'] = df_1984.infl.diff()
# df_1984['l1infl'] = df_1984.infl.shift(1)

In [ ]:
# Manual unit root regression for inflation:
# reg_infl = smf.ols('diff_infl ~ trend + l1infl', data=df_1984, missing='drop').fit()
# print(reg_infl.params)
# print(f't-stat on l1infl: {reg_infl.tvalues["l1infl"]:.4f}')

In [ ]:
# Automated ADF test for inflation
# adf_infl = adfuller(df_1984.infl.dropna(), maxlag=0, regression='ct')
# print(f'Test statistic: {adf_infl[0]:.4f}')
# print(f'p-value:        {adf_infl[1]:.6f}')
# print(f'Critical values: {adf_infl[4]}')

**Key finding:** CPI has a unit root (non-stationary), but inflation does not (stationary). This is intuitive: prices trend upward, but the *rate* of price change tends to fluctuate around a mean.

---

## Part 3: Pseudo Out-of-Sample Forecasting

### Exercise 5: Evaluate Forecast Performance

**Algorithm for pseudo out-of-sample forecasts:**

1. Pretend data ends at period $s$ (e.g., 2004:Q4)
2. Estimate AR(1) using data up to $s$
3. Forecast for $s+1$
4. Compare forecast to actual value
5. Shift $s$ forward by one period, repeat

This simulates real-time forecasting and helps evaluate model performance.

**Run the helper function below:**

In [ ]:
def ar1_pseudo(input_df, y, startdate):
    """
    Generate pseudo out-of-sample forecasts and errors from AR(1) model.
    
    Parameters:
    - input_df: DataFrame with quarterly index
    - y: name of variable to forecast
    - startdate: first quarter to forecast (e.g., '2005Q1')
    
    Returns: DataFrame with forecasts and forecast errors
    """
    df = pd.DataFrame()
    df['y'] = input_df[y]
    df['lag_y'] = df.y.shift(1)
    df['trend'] = range(len(input_df[y]))
    df['next_forecast'] = pd.Series(dtype='float64')
    
    start_period = pd.Period(startdate, freq='Q')
    
    for i, row in df.iterrows():
        if row.name < start_period - 1:
            continue
        df_sub = df[df.index <= row.name]
        ar_tmp = smf.ols('y ~ lag_y + trend', data=df_sub, missing='drop').fit()
        df_aux = {'lag_y': row.y, 'trend': row.trend}
        prediction = ar_tmp.predict(df_aux)
        df.at[i, 'next_forecast'] = prediction[0]
    
    df.loc[df.iloc[-1].name + 1, :] = np.nan
    df['forecast'] = df.next_forecast.shift(1)
    df['forecast_error'] = df.y - df.forecast
    
    results_df = df[['forecast', 'forecast_error']].copy()
    results_df['date'] = results_df.index.to_timestamp()
    return results_df

In [ ]:
# Generate pseudo out-of-sample forecasts
# results = ar1_pseudo(df_1984, 'infl', '2005Q1')
# results.tail(10)

In [ ]:
# Plot forecasts vs actual inflation
# fig, ax = plt.subplots(figsize=(10, 5))
# df_plot = df_1984[df_1984.index >= '2000Q1']
# ax.plot(df_plot.date, df_plot.infl, 'g-', linewidth=2, label='Actual')
# ax.plot(results.date, results.forecast, 'b--', linewidth=1, label='AR(1) Forecast')
# ax.set_xlabel('Date')
# ax.set_ylabel('Inflation (%)')
# ax.set_title('Pseudo Out-of-Sample Forecasts vs Actual')
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.show()

In [ ]:
# Plot forecast errors
# fig, ax = plt.subplots(figsize=(10, 4))
# ax.plot(results.date, results.forecast_error, 'b-')
# ax.axhline(0, color='red', linestyle='--')
# ax.set_xlabel('Date')
# ax.set_ylabel('Forecast Error (%)')
# ax.set_title('AR(1) Forecast Errors')
# ax.grid(True, alpha=0.3)
# plt.show()

In [ ]:
# Summary statistics of forecast errors
# print(f'Mean forecast error:  {results.forecast_error.mean():.4f}')
# print(f'Std deviation:        {results.forecast_error.std():.4f}')
# print(f'Variance:             {results.forecast_error.var():.4f}')
# print(f'RMSE:                 {np.sqrt((results.forecast_error**2).mean()):.4f}')

---

## Summary

### Key Concepts

**Unit Roots:**
- A unit root implies non-stationarity (stochastic trend)
- Test using ADF test, NOT standard t-distribution
- `adfuller(series, maxlag=0, regression='ct')` for series with trend

**Our Findings:**
- CPI: Unit root (non-stationary) - cannot reject $H_0$
- Inflation: No unit root (stationary) - reject $H_0$

**Pseudo Out-of-Sample Forecasting:**
- Simulates real-time forecasting
- Better evaluation of predictive ability than in-sample fit
- Forecast error = actual - forecast

### Implications

- Model CPI in first differences (or use inflation directly)
- Inflation can be modeled in levels with AR models
- Simple AR(1) provides reasonable forecasts, but errors can be substantial